# Phase 8 (follow-up) — tuning `top_k` and `temperature`

Two untested defaults surfaced from `scripts/09. evaluation_harness.ipynb`'s failure analysis:

1. **`top_k=5`** for the generator's retrieved context — never actually A/B tested for the generation stage, just inherited from the `Retriever.retrieve()` default. 3 of the 40-question harness's 7 citation misses (q25, q29, q31) were cases where the correct article was retrieved at rank 6–7 — inside recall@10 (0.900) but outside recall@5 (0.825). Raising `top_k` should fix those directly, at the cost of more context tokens and more chances for the model to grab an irrelevant chunk.
2. **`temperature`** — never set at all, so every generation call so far ran at the provider default (~1.0). For a legal-QA task where consistency matters, a lower temperature is worth testing.

This notebook runs a 2×2 grid — `top_k` ∈ {5, 10} × `temperature` ∈ {default (unset), 0.0} — across all 40 eval questions, scored the same way as the main harness (regex citation proxy + LLM-as-judge faithfulness/relevancy, Claude Sonnet 5 via OpenRouter). Retrieval itself doesn't change between configs (same `Retriever`), only what's handed to the generator and how it's sampled.

Requires `COHERE_API_KEY`, `PINECONE_API_KEY`, `OPENROUTER_API_KEY` in `.env`. Kernel: `rag311`.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.retrieval import Retriever
from src.generation import get_production_generator, ARTICLE_NUM_RE

retriever = Retriever()

EVAL_PATH = PROJECT / "eval" / "eval_draft.jsonl"
eval_questions = pd.DataFrame([json.loads(l) for l in open(EVAL_PATH, encoding="utf-8")])

CONFIGS = [
    {"name": "top5_default-temp", "top_k": 5, "temperature": None},
    {"name": "top5_temp0", "top_k": 5, "temperature": 0.0},
    {"name": "top10_default-temp", "top_k": 10, "temperature": None},
    {"name": "top10_temp0", "top_k": 10, "temperature": 0.0},
]
print(f"{len(eval_questions)} questions x {len(CONFIGS)} configs = {len(eval_questions) * len(CONFIGS)} generations")

40 questions x 4 configs = 160 generations


## Generate across the grid

Retrieval is cached per `top_k` value (2 retrieve calls per question instead of 4) since the semantic/keyword candidate lists don't depend on temperature.

In [2]:
chunks_cache = {}  # (question_id, top_k) -> chunks

def get_chunks(question_text, question_id, top_k):
    key = (question_id, top_k)
    if key not in chunks_cache:
        chunks_cache[key] = retriever.retrieve(question_text, top_k=top_k)
    return chunks_cache[key]

rows = []
answers = {}  # (question_id, config_name) -> answer
chunks_by_qid_config = {}  # (question_id, config_name) -> chunks used

for cfg in CONFIGS:
    gen = get_production_generator(temperature=cfg["temperature"])
    for row in eval_questions.itertuples(index=False):
        chunks = get_chunks(row.question_text, row.question_id, cfg["top_k"])
        available_articles = {c["article_number"] for c in chunks}
        correct_articles = set(row.answer_article_numbers)

        result = gen.generate(row.question_text, chunks)
        answer = result["answer"]
        answers[(row.question_id, cfg["name"])] = answer
        chunks_by_qid_config[(row.question_id, cfg["name"])] = chunks

        cited_articles = {g1 or g2 for g1, g2 in ARTICLE_NUM_RE.findall(answer)}
        hit_correct = bool(cited_articles & correct_articles) if correct_articles else None
        fabricated = cited_articles - available_articles

        rows.append({
            "config": cfg["name"], "top_k": cfg["top_k"], "temperature": cfg["temperature"],
            "question_id": row.question_id,
            "cited_articles": sorted(cited_articles),
            "hit_correct_article": hit_correct,
            "fabricated_citation": bool(fabricated),
        })
    print(f"done: {cfg['name']}")

tuning_results = pd.DataFrame(rows)
tuning_results.to_csv(PROJECT / "eval" / "config_tuning_results.csv", index=False, encoding="utf-8-sig")
print(tuning_results.groupby("config").agg(hit_rate=("hit_correct_article", "mean"), fabrication_rate=("fabricated_citation", "mean")).round(3))

done: top5_default-temp


done: top5_temp0


done: top10_default-temp


done: top10_temp0
                    hit_rate  fabrication_rate
config                                        
top10_default-temp     0.825             0.025
top10_temp0            0.850             0.050
top5_default-temp      0.825             0.025
top5_temp0             0.775             0.050


## LLM-as-judge across the grid

Same judge/prompt as the main harness (Claude Sonnet 5 via OpenRouter), run on all 160 (question, config) answers.

In [3]:
import re as _re
import time as _time
import os

from openai import OpenAI

judge_client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")

import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from evallib import (JUDGE_MODEL, JUDGE_PROMPT,
                     format_context_for_judge, make_judge)

judge = make_judge(judge_client)
judge = make_judge(judge_client)




judge_rows = []
for cfg in CONFIGS:
    for row in eval_questions.itertuples(index=False):
        chunks = chunks_by_qid_config[(row.question_id, cfg["name"])]
        answer = answers[(row.question_id, cfg["name"])]
        score = judge(row.question_text, chunks, answer)
        judge_rows.append({"config": cfg["name"], "question_id": row.question_id, **score})
    print(f"judged: {cfg['name']}")

tuning_judge_results = pd.DataFrame(judge_rows)
tuning_judge_results.to_csv(PROJECT / "eval" / "config_tuning_judge_scores.csv", index=False, encoding="utf-8-sig")
print("done,", len(tuning_judge_results), "judge scores")

judged: top5_default-temp


judged: top5_temp0


judged: top10_default-temp


judged: top10_temp0
done, 160 judge scores


In [4]:
summary = tuning_results.groupby("config").agg(
    hit_rate=("hit_correct_article", "mean"),
    fabrication_rate=("fabricated_citation", "mean"),
).join(
    tuning_judge_results.groupby("config").agg(
        mean_faithfulness=("faithfulness_score", "mean"),
        min_faithfulness=("faithfulness_score", "min"),
        mean_relevancy=("relevancy_score", "mean"),
        min_relevancy=("relevancy_score", "min"),
    )
).reindex([c["name"] for c in CONFIGS])

print(summary.round(3).to_string())

                    hit_rate  fabrication_rate  mean_faithfulness  min_faithfulness  mean_relevancy  min_relevancy
config                                                                                                            
top5_default-temp      0.825             0.025              4.825                 2           4.675              2
top5_temp0             0.775             0.050              4.775                 1           4.800              2
top10_default-temp     0.825             0.025              4.900                 4           4.850              3
top10_temp0            0.850             0.050              4.925                 4           4.900              4
